# NLU messages (Phase 1d) — L40 + vLLM

Generates labelled user messages for the 7 intents (`request_diss`, `another_one`, `make_meaner`, `make_softer`, `greet`, `goodbye`, `out_of_scope`). Messages that mention traits come with the exact words that express each trait, which `prepare_nlu.py` turns into BIO slot tags. Messages about held-out traits go to the NLU test set only. About 3,000 messages before cleaning.

**Upload next to this notebook:** `traits.json` from `data/` in the repo.

**Workflow:** `MODE = "pilot"` → Run All → read the samples → `MODE = "full"` → Restart & Run All → copy `llm_out/nlu.jsonl` and `llm_out/nlu_meta.json` to `data/llm_out/` in the repo → `python data/prepare_nlu.py`.

**Model:** Qwen/Qwen3-32B-FP8 (Apache-2.0).

## 1. Install

In [1]:
%pip install -q vllm


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Settings

In [2]:
import os, re, json, math, time, random, itertools
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"   # skip FlashInfer's sampler, which needs nvcc >= 12.8 to compile
from collections import Counter, defaultdict
from datetime import date

# ======================= THE SWITCH =======================
MODE = "full"              # "pilot" = a few requests to read the output | "full" = the real run
# ==========================================================
assert MODE in ("pilot", "full")

MODEL_ID = "Qwen/Qwen3-32B-FP8"           # same model as the quatrain corpus; thinking switched off below
CHAT_KW  = {"chat_template_kwargs": {"enable_thinking": False}}
SEED, TEMPERATURE, TOP_P = 42, 0.9, 0.95
MAX_TOKENS = 2048

NLU_PER_REQUEST     = 20       # messages per request
NLU_TRAIT_REQUESTS  = 2        # per trait (training and held-out), each in a different style
NLU_PAIRS, NLU_PER_PAIR = 60, 5
NLU_INTENT_REQUESTS = {"request_diss": 8, "another_one": 15, "make_meaner": 15, "make_softer": 15,
                       "greet": 10, "goodbye": 10, "out_of_scope": 15}   # request_diss here = no trait given

OUT_DIR = "llm_out" if MODE == "full" else "llm_pilot_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("MODE:", MODE, "| output:", OUT_DIR)

MODE: full | output: llm_out


## 3. Traits

In [3]:
TRAITS  = json.load(open("traits.json", encoding="utf-8"))
TRAIN   = [t for t, v in TRAITS.items() if not v["heldout"]]
HELDOUT = [t for t, v in TRAITS.items() if v["heldout"]]
print(f"{len(TRAIN)} training traits | held out: {HELDOUT}")

35 training traits | held out: ['eloquent', 'punctual', 'patient', 'worldly', 'frugal']


## 4. Load the model

If it runs out of memory, lower `gpu_memory_utilization` to 0.85.

In [4]:
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL_ID, max_model_len=4096, gpu_memory_utilization=0.90,
          enable_prefix_caching=True, seed=SEED)
params = SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS, repetition_penalty=1.05)

def chat(msgs):
    # one reply per conversation -> (texts, output tokens)
    if not msgs:
        return [], 0
    outs = llm.chat(msgs, params, use_tqdm=True, **CHAT_KW)
    return [o.outputs[0].text for o in outs], sum(len(o.outputs[0].token_ids) for o in outs)

/common/home/projectgrps/CS480/CS480G3/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(MainProcess pid=446361) INFO 10-29 18:11:04 [api_utils.py:286] non-default args: {'seed': 42, 'max_model_len': 4096, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-32B-FP8'}
(MainProcess pid=446361) INFO 10-29 18:11:05 [model.py:733] Resolved architecture: Qwen3ForCausalLM
(MainProcess pid=446361) INFO 10-29 18:11:05 [model.py:2152] Using max model len 4096
(MainProcess pid=446361) INFO 10-29 18:11:07 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=446361) INFO 10-29 18:11:07 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(MainProcess pid=446361) INFO 10-29 18:11:07 [compilation.py:317] Enabled custom fusions: norm_quant, act_quant


Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 1238.57it/s]


(EngineCore pid=446446) INFO 10-29 18:11:10 [core.py:133] Initializing a V1 LLM engine (v0.32.0) with config: model='Qwen/Qwen3-32B-FP8', speculative_config=None, tokenizer='Qwen/Qwen3-32B-FP8', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=fp8, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidde

Loading safetensors checkpoint shards:   0% Completed | 0/7 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  14% Completed | 1/7 [00:05<00:31,  5.30s/it]
Loading safetensors checkpoint shards:  29% Completed | 2/7 [00:10<00:26,  5.38s/it]
Loading safetensors checkpoint shards:  43% Completed | 3/7 [00:16<00:21,  5.37s/it]
Loading safetensors checkpoint shards:  57% Completed | 4/7 [00:21<00:16,  5.37s/it]
Loading safetensors checkpoint shards:  71% Completed | 5/7 [00:26<00:10,  5.38s/it]
Loading safetensors checkpoint shards:  86% Completed | 6/7 [00:32<00:05,  5.39s/it]
Loading safetensors checkpoint shards: 100% Completed | 7/7 [00:37<00:00,  5.36s/it]
Loading safetensors checkpoint shards: 100% Completed | 7/7 [00:37<00:00,  5.37s/it]



(EngineCore pid=446446) INFO 10-29 18:11:52 [default_loader.py:501] Loading weights took 37.71 seconds
(EngineCore pid=446446) WARNING 10-29 18:11:53 [marlin_utils_fp8.py:108] Your GPU does not have native support for FP8 computation but FP8 quantization is being used. Weight-only FP8 compression will be used leveraging the Marlin kernel. This may degrade performance for compute-heavy workloads.
(EngineCore pid=446446) INFO 10-29 18:11:54 [model_runner.py:424] Model loading took 32.52 GiB memory and 43.165427 seconds
(EngineCore pid=446446) INFO 10-29 18:11:55 [topk_topp_sampler.py:96] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=446446) INFO 10-29 18:11:55 [torch_utils.py:295] Reducing Torch threads from 4 to 1 for serving. Set OMP_NUM_THREADS in the external environment to override.
(EngineCore pid=446446) INFO 10-29 18:11:55 [utils.py:347] Using LBNHC KV cache layout.
(EngineCore pid=446446) INFO 10-29 18:11:57 [caching.py:342] reconstr

Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00,  3.24it/s]


(EngineCore pid=446446) INFO 10-29 18:12:23 [model_runner.py:1127] Graph capturing finished in 18 secs, took 0.73 GiB
(EngineCore pid=446446) INFO 10-29 18:12:24 [gpu_worker.py:759] Available KV cache memory: 4.89 GiB
(EngineCore pid=446446) INFO 10-29 18:12:24 [gpu_worker.py:774] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9000 is equivalent to --gpu-memory-utilization=0.8820 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9180. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=446446) INFO 10-29 18:12:24 [kv_cache_utils.py:2327] GPU KV cache size: 20,016 tokens, Maximum concurrency for 4,096 tokens per request: 4.89x
(EngineCore pid=446446) INFO 10-29 18:12:24 [kernel_warmup.py:176] JIT kernel warmup starting.
(EngineCore pid=446446) INFO 10-29 18:12:24 [kernel_warmup.py:189] JIT kernel warmup finished in 0.00s.


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:06<00:00,  5.14it/s]


(EngineCore pid=446446) INFO 10-29 18:12:49 [model_runner.py:1127] Graph capturing finished in 24 secs, took 0.42 GiB
(EngineCore pid=446446) INFO 10-29 18:12:49 [gpu_worker.py:971] CUDA graph pool memory: 0.42 GiB (actual), 0.8 GiB (estimated), difference: 0.38 GiB (92.5%).
(EngineCore pid=446446) INFO 10-29 18:12:49 [gpu_worker.py:1034] Free memory on device (44.16/44.42 GiB) on startup. Desired GPU memory utilization is (0.9, 39.98 GiB). Actual usage is 32.85 GiB for consumed memory (weights + non-torch), 2.25 GiB for peak activation, and 0.42 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=4646365696` (4.33 GiB) to fit into requested memory, or `--kv-cache-memory=9134402048` (8.51 GiB) to fully utilize gpu memory. Current kv cache memory in use is 4.89 GiB.
(EngineCore pid=446446) INFO 10-29 18:12:51 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=446446) INFO 1

## 5. Prompts

Trait messages mix direct mentions with indirect descriptions, and every request uses one of six writing styles so the messages are not all alike.

In [5]:
QUOTE_MAP = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"'})

def normalize(text):
    return text.translate(QUOTE_MAP)        # curly quotes -> straight, as in form_checks.py

THINK_RE = re.compile(r"<think>.*?</think>", re.S)

def parse_json_array(text):
    m = re.search(r"\[.*\]", THINK_RE.sub("", text), re.S)
    try:
        v = json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None
    return v if isinstance(v, list) else None

def dump(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)

STYLES = ["casual texting: lowercase, slang and a typo or two", "polite, complete sentences",
          "very short, 2 to 6 words", "Gen Z slang", "rambling, with extra details about their day",
          "written by a non-native English speaker"]
NLU_SYSTEM = "You write realistic chat messages that users send to a chatbot. Reply with a JSON array only."
INTENT_DESC = {
    "request_diss": "ask the bot to roast them, without saying anything about themselves",
    "another_one":  "already got a roast and ask for another one",
    "make_meaner":  "already got a roast and ask for a harsher, more savage one",
    "make_softer":  "already got a roast and ask for a gentler, nicer one",
    "greet":        "greet the bot at the start of a chat",
    "goodbye":      "end the chat or say goodbye",
    "out_of_scope": "ask the roast bot for something unrelated, like the weather, homework help, a recipe or a fact",
}

def nlu_trait_prompt(traits, k, style):
    desc = " and ".join(f'"{t}" (e.g. {", ".join(TRAITS[t]["synonyms"])})' for t in traits)
    return (f"Write {k} different messages a user might send to a roast-poem chatbot, asking to be roasted "
            f"and revealing that they are {desc}. Each message is about the user themself (\"I\", \"me\").\n"
            f"Style: {style}.\n"
            'Mix direct mentions ("roast me, I\'m super organized") with indirect ones that only describe the '
            'trait ("people say I never shut up about history"). Do not always use the words given above.\n'
            'Return a JSON array of objects: {"text": the message, "spans": [{"trait": '
            + " or ".join(f'"{t}"' for t in traits)
            + ', "span": the exact words in the text that express it}]}. Copy every span exactly from the text.')

def trait_name(label, traits):
    # the model often labels a span with a synonym ("clever") instead of the trait ("smart")
    label = str(label).strip().lower()
    for t in traits:
        if label == t or label in (w.lower() for w in TRAITS[t]["synonyms"]):
            return t
    return traits[0] if len(traits) == 1 else label

def nlu_intent_prompt(intent, k, style):
    return (f"Write {k} different messages where a user of a roast-poem chatbot would {INTENT_DESC[intent]}.\n"
            f"Style: {style}.\nReturn a JSON array of strings.")

print(nlu_trait_prompt(["knowledgeable"], NLU_PER_REQUEST, STYLES[0]))

Write 20 different messages a user might send to a roast-poem chatbot, asking to be roasted and revealing that they are "knowledgeable" (e.g. well-informed, a walking encyclopedia, knows a lot, into history). Each message is about the user themself ("I", "me").
Style: casual texting: lowercase, slang and a typo or two.
Mix direct mentions ("roast me, I'm super organized") with indirect ones that only describe the trait ("people say I never shut up about history"). Do not always use the words given above.
Return a JSON array of objects: {"text": the message, "spans": [{"trait": "knowledgeable", "span": the exact words in the text that express it}]}. Copy every span exactly from the text.


## 6. Run

In [6]:
def nlu_jobs():
    rng, jobs = random.Random(SEED + 2), []
    msg = lambda p: [{"role": "system", "content": NLU_SYSTEM}, {"role": "user", "content": p}]
    trait_list = TRAIN[:2] + HELDOUT[:1] if MODE == "pilot" else TRAIN + HELDOUT
    for t in trait_list:
        for style in rng.sample(STYLES, NLU_TRAIT_REQUESTS):
            jobs.append(("request_diss", [t], style, msg(nlu_trait_prompt([t], NLU_PER_REQUEST, style))))
    for p in rng.sample(list(itertools.combinations(TRAIN, 2)), 2 if MODE == "pilot" else NLU_PAIRS):
        style = rng.choice(STYLES)
        jobs.append(("request_diss", list(p), style, msg(nlu_trait_prompt(list(p), NLU_PER_PAIR, style))))
    for intent, n in NLU_INTENT_REQUESTS.items():
        for _ in range(1 if MODE == "pilot" else n):
            style = rng.choice(STYLES)
            jobs.append((intent, [], style, msg(nlu_intent_prompt(intent, NLU_PER_REQUEST, style))))
    return jobs

jobs, t0 = nlu_jobs(), time.time()
print(f"{len(jobs)} requests")
texts, tokens = chat([m for *_, m in jobs])
nlu_rows, bad = [], Counter()
for (intent, traits, style, _), text in zip(jobs, texts):
    items = parse_json_array(text)
    if items is None:
        bad["bad_json"] += 1
        continue
    for it in items:
        if traits:
            if not (isinstance(it, dict) and isinstance(it.get("text"), str)):
                bad["bad_item"] += 1; continue
            spans = [{"trait": trait_name(s.get("trait"), traits), "span": normalize(str(s.get("span") or ""))}
                     for s in it.get("spans") or [] if isinstance(s, dict)]
            text_, group = it["text"], "heldout" if any(TRAITS[t]["heldout"] for t in traits) else "train"
        else:
            if not isinstance(it, str):
                bad["bad_item"] += 1; continue
            text_, spans, group = it, [], "train"
        nlu_rows.append({"intent": intent, "text": normalize(text_).strip(), "spans": spans,
                         "traits": traits, "group": group, "style": style})
dump(f"{OUT_DIR}/nlu.jsonl", nlu_rows)
stats = {"messages": len(nlu_rows), "by_intent": dict(Counter(r["intent"] for r in nlu_rows)),
         "problems": dict(bad), "tokens": tokens, "seconds": round(time.time() - t0, 1)}
print(f"{len(nlu_rows):,} messages | {stats['by_intent']} | problems {dict(bad)}")

228 requests


Processed prompts: 100%|██████████| 228/228 [07:51<00:00,  2.07s/it, est. speed input: 76.75 toks/s, output: 298.53 toks/s]

3,579 messages | {'request_diss': 2019, 'another_one': 280, 'make_meaner': 300, 'make_softer': 300, 'greet': 180, 'goodbye': 200, 'out_of_scope': 300} | problems {'bad_json': 4}


## 7. Samples

Check that the spans really are the words that express the trait, and that the messages sound like real users.

In [7]:
for r in random.Random().sample(nlu_rows, min(12, len(nlu_rows))):
    print(f"  [{r['intent']}] {r['text']}   {r['spans'] or ''}")

  [request_diss] roast me bc i know more than the teacher half the time   [{'trait': 'knowledgeable', 'span': 'know more than the teacher'}]
  [request_diss] You can call me Mr. Well-Read, but I promise I won't correct your grammar while you're roasting me.   [{'trait': 'knowledgeable', 'span': 'Mr. Well-Read'}]
  [request_diss] roast me for over-listening   [{'trait': 'caring', 'span': 'over-listening'}]
  [greet] Hello again! Can't wait to see what clever lines you'll come up with.   
  [request_diss] i track grocery lists like they're stock portfolios   [{'trait': 'organized', 'span': 'track grocery lists'}]
  [make_softer] Go easy on me.   
  [greet] Hello, what is needed to start a roast poem?   
  [out_of_scope] Explain photosynthesis.   
  [make_meaner] I like what I read, but can we take it to the next level of harshness?   
  [request_diss] Roast me for knowing the difference between saffron from India and Spain.   [{'trait': 'worldly', 'span': 'knowing the difference between 

## 8. Save the run record

In [8]:
meta = {"date": date.today().isoformat(), "mode": MODE, "model": MODEL_ID, "license": "Apache-2.0",
        "temperature": TEMPERATURE, "top_p": TOP_P, "seed": SEED,
        "settings": {"NLU_PER_REQUEST": NLU_PER_REQUEST, "NLU_TRAIT_REQUESTS": NLU_TRAIT_REQUESTS, "NLU_PAIRS": NLU_PAIRS,
                     "NLU_PER_PAIR": NLU_PER_PAIR, "NLU_INTENT_REQUESTS": NLU_INTENT_REQUESTS, "STYLES": STYLES},
        "prompts": {"system": NLU_SYSTEM, "trait_example": nlu_trait_prompt(["confident"], NLU_PER_REQUEST, STYLES[0]),
                    "intent_example": nlu_intent_prompt("make_meaner", NLU_PER_REQUEST, STYLES[0])},
        "stats": stats}
json.dump(meta, open(f"{OUT_DIR}/nlu_meta.json", "w", encoding="utf-8"), indent=2)
print("Saved to", OUT_DIR)

Saved to llm_out
